# Extração de Blendshapes e Face Mesh — MediaPipe
52 blendshapes + 478 landmarks por frame → .npy por vídeo

In [ ]:
# CÉLULA 0: Instalar dependências
!pip install mediapipe opencv-python

In [1]:
# CÉLULA 1: Imports e configuração
import os
import re
import time
import numpy as np
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from tqdm.notebook import tqdm

# === AJUSTE ESSES PATHS ===
#DIRECTORY_VARIABLES NEED TO CHANGE TO YOUR CORRECT DIRECTORIES
BASE_DIR = r'\ABAW\data'

FACES_DIR = os.path.join(BASE_DIR, 'cropped-aligned-faces', 'Videos')
OUTPUT_BS_DIR = os.path.join(BASE_DIR, 'blendshape_features')   # 52 blendshapes
OUTPUT_MESH_DIR = os.path.join(BASE_DIR, 'mesh_features')       # 478 landmarks (x,y,z)

os.makedirs(OUTPUT_BS_DIR, exist_ok=True)
os.makedirs(OUTPUT_MESH_DIR, exist_ok=True)

# Amostragem (mesmo rate da extração de AUs)
SAMPLE_RATE = 3  # 1 a cada 3 frames → ~10fps

print(f"Faces dir: {FACES_DIR}")
print(f"Blendshapes output: {OUTPUT_BS_DIR}")
print(f"Mesh output: {OUTPUT_MESH_DIR}")

Faces dir: C:\Users\ddonz\OneDrive\Documentos\Aislan\data\cropped-aligned-faces\Videos
Blendshapes output: C:\Users\ddonz\OneDrive\Documentos\Aislan\data\blendshape_features
Mesh output: C:\Users\ddonz\OneDrive\Documentos\Aislan\data\mesh_features


In [2]:
# CÉLULA 2: Baixar modelo e inicializar FaceLandmarker
import urllib.request

MODEL_PATH = os.path.join(BASE_DIR, 'face_landmarker.task')

if not os.path.exists(MODEL_PATH):
    print("Baixando modelo FaceLandmarker...")
    url = 'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task'
    urllib.request.urlretrieve(url, MODEL_PATH)
    print("Download concluído!")
else:
    print("Modelo já existe.")

# Configurar FaceLandmarker
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=True,
    output_facial_transformation_matrixes=False,
    num_faces=1
)
detector = vision.FaceLandmarker.create_from_options(options)
print("FaceLandmarker pronto!")

Modelo já existe.
FaceLandmarker pronto!


In [3]:
# CÉLULA 3: Testar em 1 frame pra verificar output

# Pegar um frame de teste
test_pid = sorted(os.listdir(FACES_DIR))[0]
test_visit = os.path.join(FACES_DIR, test_pid, 'Visite_1')
test_video = sorted(os.listdir(test_visit))[0]
test_path = os.path.join(test_visit, test_video)
test_frame = sorted([f for f in os.listdir(test_path) if f.endswith('.jpg')])[0]
test_frame_path = os.path.join(test_path, test_frame)

# Processar
image = mp.Image.create_from_file(test_frame_path)
result = detector.detect(image)

if result.face_blendshapes and len(result.face_blendshapes) > 0:
    bs = result.face_blendshapes[0]
    bs_names = [b.category_name for b in bs]
    bs_values = [b.score for b in bs]
    print(f"Blendshapes: {len(bs)} categorias")
    print(f"Nomes: {bs_names[:10]}...")
    print(f"Valores: {[f'{v:.3f}' for v in bs_values[:10]]}...")
else:
    print("AVISO: Nenhum blendshape detectado!")

if result.face_landmarks and len(result.face_landmarks) > 0:
    landmarks = result.face_landmarks[0]
    print(f"\nLandmarks: {len(landmarks)} pontos (x, y, z)")
    print(f"Primeiro landmark: ({landmarks[0].x:.4f}, {landmarks[0].y:.4f}, {landmarks[0].z:.4f})")
else:
    print("AVISO: Nenhum landmark detectado!")

Blendshapes: 52 categorias
Nomes: ['_neutral', 'browDownLeft', 'browDownRight', 'browInnerUp', 'browOuterUpLeft', 'browOuterUpRight', 'cheekPuff', 'cheekSquintLeft', 'cheekSquintRight', 'eyeBlinkLeft']...
Valores: ['0.000', '0.079', '0.071', '0.017', '0.045', '0.031', '0.000', '0.000', '0.000', '0.298']...

Landmarks: 478 pontos (x, y, z)
Primeiro landmark: (0.4997, 0.7634, -0.1257)


In [4]:
# CÉLULA 4: Listar nomes dos 52 blendshapes pra referência
if result.face_blendshapes and len(result.face_blendshapes) > 0:
    bs = result.face_blendshapes[0]
    print(f"Total: {len(bs)} blendshapes\n")
    for i, b in enumerate(bs):
        print(f"  {i:2d}. {b.category_name:<30s} = {b.score:.4f}")
    
    # Salvar nomes pra uso futuro
    BS_NAMES = [b.category_name for b in bs]
    N_BS = len(BS_NAMES)
    print(f"\nN_BS = {N_BS}")

Total: 52 blendshapes

   0. _neutral                       = 0.0000
   1. browDownLeft                   = 0.0791
   2. browDownRight                  = 0.0715
   3. browInnerUp                    = 0.0171
   4. browOuterUpLeft                = 0.0453
   5. browOuterUpRight               = 0.0310
   6. cheekPuff                      = 0.0000
   7. cheekSquintLeft                = 0.0000
   8. cheekSquintRight               = 0.0000
   9. eyeBlinkLeft                   = 0.2978
  10. eyeBlinkRight                  = 0.2576
  11. eyeLookDownLeft                = 0.3801
  12. eyeLookDownRight               = 0.3575
  13. eyeLookInLeft                  = 0.0266
  14. eyeLookInRight                 = 0.1960
  15. eyeLookOutLeft                 = 0.1853
  16. eyeLookOutRight                = 0.0149
  17. eyeLookUpLeft                  = 0.0241
  18. eyeLookUpRight                 = 0.0250
  19. eyeSquintLeft                  = 0.2688
  20. eyeSquintRight                 = 0.3087
  21. eyeWi

In [5]:
# CÉLULA 5: Descobrir todos os vídeos e checar resume

def frame_sort_key(filename):
    match = re.search(r'frame-(\d+)', filename)
    return int(match.group(1)) if match else 0

all_videos = []
for pid in sorted(os.listdir(FACES_DIR)):
    visit_path = os.path.join(FACES_DIR, pid, 'Visite_1')
    if not os.path.isdir(visit_path):
        continue
    for video_name in sorted(os.listdir(visit_path)):
        v_path = os.path.join(visit_path, video_name)
        if os.path.isdir(v_path):
            all_videos.append((pid, video_name, v_path))

# Checar resume
to_process = []
already_done = 0
for pid, video_name, v_path in all_videos:
    bs_path = os.path.join(OUTPUT_BS_DIR, pid, f"{video_name}.npy")
    if os.path.exists(bs_path):
        already_done += 1
    else:
        to_process.append((pid, video_name, v_path))

print(f"Total de vídeos: {len(all_videos)}")
print(f"Já processados: {already_done}")
print(f"A processar: {len(to_process)}")

Total de vídeos: 1427
Já processados: 0
A processar: 1427


In [6]:
# CÉLULA 6: Extração principal — blendshapes + mesh landmarks

SAVE_MESH = True  # Setar False se quiser só blendshapes (mesh ocupa mais espaço)

errors = []
total_frames = 0
start_time = time.time()

for idx, (pid, video_name, v_path) in enumerate(tqdm(to_process, desc="Vídeos")):
    # Preparar output dirs
    bs_out_dir = os.path.join(OUTPUT_BS_DIR, pid)
    os.makedirs(bs_out_dir, exist_ok=True)
    bs_out_path = os.path.join(bs_out_dir, f"{video_name}.npy")
    
    if SAVE_MESH:
        mesh_out_dir = os.path.join(OUTPUT_MESH_DIR, pid)
        os.makedirs(mesh_out_dir, exist_ok=True)
        mesh_out_path = os.path.join(mesh_out_dir, f"{video_name}.npy")
    
    # Listar, ordenar e amostrar frames
    frames = [f for f in os.listdir(v_path) if f.endswith(('.jpg', '.png'))]
    frames = sorted(frames, key=frame_sort_key)[::SAMPLE_RATE]
    
    if len(frames) == 0:
        errors.append(f"{pid}/{video_name}: sem frames")
        continue
    
    bs_values = []
    mesh_values = []
    n_failed = 0
    
    for frame_file in frames:
        frame_path = os.path.join(v_path, frame_file)
        try:
            image = mp.Image.create_from_file(frame_path)
            result = detector.detect(image)
            
            # Blendshapes
            if result.face_blendshapes and len(result.face_blendshapes) > 0:
                bs = [b.score for b in result.face_blendshapes[0]]
                bs_values.append(bs)
            else:
                bs_values.append([0.0] * N_BS)
                n_failed += 1
            
            # Mesh landmarks
            if SAVE_MESH:
                if result.face_landmarks and len(result.face_landmarks) > 0:
                    lm = [[l.x, l.y, l.z] for l in result.face_landmarks[0]]
                    mesh_values.append(lm)
                else:
                    mesh_values.append([[0.0, 0.0, 0.0]] * 478)
        except Exception as e:
            bs_values.append([0.0] * N_BS)
            if SAVE_MESH:
                mesh_values.append([[0.0, 0.0, 0.0]] * 478)
            n_failed += 1
    
    # Salvar
    bs_array = np.array(bs_values, dtype=np.float32)  # (N_frames, 52)
    np.save(bs_out_path, bs_array)
    
    if SAVE_MESH:
        mesh_array = np.array(mesh_values, dtype=np.float32)  # (N_frames, 478, 3)
        np.save(mesh_out_path, mesh_array)
    
    total_frames += len(frames)
    
    if n_failed > 0:
        errors.append(f"{pid}/{video_name}: {n_failed}/{len(frames)} frames sem face")
    
    # Progresso a cada 200 vídeos
    if (idx + 1) % 200 == 0:
        elapsed = time.time() - start_time
        fps = total_frames / elapsed if elapsed > 0 else 0
        print(f"  {idx+1}/{len(to_process)} | {total_frames} frames | {fps:.0f} f/s")

elapsed = time.time() - start_time
print(f"\nConcluído em {elapsed/60:.1f} min")
print(f"Vídeos: {len(to_process)} | Frames: {total_frames} | Erros: {len(errors)}")
print(f"Velocidade: {total_frames/elapsed:.0f} frames/s")

Vídeos:   0%|          | 0/1427 [00:00<?, ?it/s]

  200/1427 | 38967 frames | 51 f/s
  400/1427 | 85042 frames | 52 f/s
  600/1427 | 132962 frames | 53 f/s
  800/1427 | 176185 frames | 53 f/s
  1000/1427 | 217408 frames | 53 f/s
  1200/1427 | 259370 frames | 53 f/s
  1400/1427 | 300716 frames | 54 f/s

Concluído em 95.3 min
Vídeos: 1427 | Frames: 306012 | Erros: 349
Velocidade: 54 frames/s


In [7]:
# CÉLULA 7: Erros (se houver)
if errors:
    print(f"Total de erros: {len(errors)}")
    for e in errors[:20]:
        print(f"  - {e}")
else:
    print("Nenhum erro!")

Total de erros: 349
  - 82563/82563_Question_1_2024-10-02_15-36-21_Video.mp4: 12/156 frames sem face
  - 82563/82563_Question_2_2024-10-02_15-42-30_Video.mp4: 12/329 frames sem face
  - 82563/82563_Question_3_2024-10-02_15-39-33_Video.mp4: 12/373 frames sem face
  - 82563/82563_Question_4_2024-10-02_15-40-55_Video.mp4: 12/276 frames sem face
  - 82563/82563_Question_5_2024-10-02_15-44-31_Video.mp4: 12/423 frames sem face
  - 82563/82563_Question_6_2024-10-02_15-47-24_Video.mp4: 12/212 frames sem face
  - 82563/82563_Question_7_2024-10-02_15-46-10_Video.mp4: 12/373 frames sem face
  - 82566/82566_Question_3_2024-10-02_15-44-42_Video.mp4: 1/267 frames sem face
  - 82568/82568_Question_1_2024-10-02_15-52-49_Video.mp4: 2/151 frames sem face
  - 82568/82568_Question_2_2024-10-02_15-55-39_Video.mp4: 2/114 frames sem face
  - 82568/82568_Question_3_2024-10-02_15-55-07_Video.mp4: 2/85 frames sem face
  - 82568/82568_Question_4_2024-10-02_15-56-17_Video.mp4: 2/128 frames sem face
  - 82568/8256

In [8]:
# CÉLULA 8: Validação

# Blendshapes
bs_files = []
for pid in os.listdir(OUTPUT_BS_DIR):
    pid_dir = os.path.join(OUTPUT_BS_DIR, pid)
    if not os.path.isdir(pid_dir):
        continue
    for f in os.listdir(pid_dir):
        if f.endswith('.npy'):
            arr = np.load(os.path.join(pid_dir, f))
            bs_files.append((pid, f, arr.shape))

print("=== BLENDSHAPES ===")
print(f"Arquivos .npy: {len(bs_files)}")
print(f"Participantes: {len(set(p for p, _, _ in bs_files))}")
frame_counts = [s[0] for _, _, s in bs_files]
print(f"Frames por vídeo: min={min(frame_counts)}, max={max(frame_counts)}, "
      f"média={np.mean(frame_counts):.0f}")
dims = set(s[1] for _, _, s in bs_files)
print(f"Dimensão: {dims}")

# NaN check
nan_count = sum(1 for p, f, _ in bs_files[:100] 
               if np.isnan(np.load(os.path.join(OUTPUT_BS_DIR, p, f))).any())
print(f"NaN nos primeiros 100: {nan_count}")

# Mesh
if SAVE_MESH:
    mesh_files = []
    for pid in os.listdir(OUTPUT_MESH_DIR):
        pid_dir = os.path.join(OUTPUT_MESH_DIR, pid)
        if not os.path.isdir(pid_dir):
            continue
        for f in os.listdir(pid_dir):
            if f.endswith('.npy'):
                arr = np.load(os.path.join(pid_dir, f))
                mesh_files.append((pid, f, arr.shape))
    
    print(f"\n=== FACE MESH ===")
    print(f"Arquivos .npy: {len(mesh_files)}")
    mesh_shapes = set(s[1:] for _, _, s in mesh_files)
    print(f"Shape landmarks: {mesh_shapes}")
    
    # Tamanho total em disco
    total_mb = sum(os.path.getsize(os.path.join(OUTPUT_MESH_DIR, p, f)) 
                   for p, f, _ in mesh_files) / 1e6
    print(f"Tamanho total mesh: {total_mb:.0f} MB")

total_bs_mb = sum(os.path.getsize(os.path.join(OUTPUT_BS_DIR, p, f)) 
                  for p, f, _ in bs_files) / 1e6
print(f"\nTamanho total blendshapes: {total_bs_mb:.0f} MB")

=== BLENDSHAPES ===
Arquivos .npy: 1427
Participantes: 300
Frames por vídeo: min=25, max=765, média=214
Dimensão: {52}
NaN nos primeiros 100: 0

=== FACE MESH ===
Arquivos .npy: 1427
Shape landmarks: {(478, 3)}
Tamanho total mesh: 1755 MB

Tamanho total blendshapes: 64 MB
